# MarginGraph: run everything (CPU)

Runs E1 to E11 from `configs/default.yaml` and shows the result tables. Every number here is read from `results/`, which `margingraph run` writes.

Put `online_retail_II.xlsx` (UCI dataset 502) in `data/raw/` first.

In [ ]:
# !git clone https://github.com/Sanskar1724/MarginGraph && %cd MarginGraph
# !pip install -q -e .
import os
os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')

In [ ]:
!margingraph prepare
!margingraph run

In [ ]:
import json, pandas as pd
R = 'results'
print(json.dumps(json.load(open(f'{R}/e1_audit.json')), indent=2))
pd.read_csv(f'{R}/e1_top_pairs.csv').head(10)

## E3 baselines and E4 retrieval (test split)

In [ ]:
e3 = pd.read_csv(f'{R}/e3_baselines.csv')
e3.query("split=='test'").pivot(index='model', columns='metric', values='mean').round(4)

In [ ]:
pd.read_csv(f'{R}/e4_retrieval.csv').query("split=='test'").round(4)

## E5 LightGBM reranker vs best baseline

In [ ]:
display(pd.read_csv(f'{R}/e5_rerank.csv').round(4))
display(pd.read_csv(f'{R}/e5_paired.csv').round(4))
pd.read_csv(f'{R}/e5_importance.csv').head(15)

## E7 calibration, E8 strategies, E9 solver check

Stage 2 uses **simulated** cost and stock (Online Retail II has neither).

In [ ]:
print(open(f'{R}/e7_calibration.json').read())
e8 = pd.read_csv(f'{R}/e8_strategies_simulated_erp.csv')
e8[['strategy','hit','realized_profit','expected_profit','overstock_hit','violations']].round(4)

In [ ]:
import matplotlib.pyplot as plt
c = e8[e8.strategy.str.match(r'milp_lam[0-9.]+$')]
fig, ax = plt.subplots(figsize=(5,4))
ax.plot(c['hit'], c['expected_profit'], marker='o')
for _, r in c.iterrows(): ax.annotate(r.strategy.replace('milp_',''), (r.hit, r.expected_profit))
ax.set_xlabel('Hit@2 (real held-out item)'); ax.set_ylabel('Expected profit per basket (simulated cost)')
ax.set_title('Relevance vs profit trade-off'); plt.show()

In [ ]:
display(pd.read_csv(f'{R}/e8_paired_vs_ranker_top2.csv').round(4))
print(open(f'{R}/e9_solver_check.json').read())

## E10 inventory simulation and E11 ERP sensitivity

In [ ]:
display(pd.read_csv(f'{R}/e10_inventory_summary.csv', header=[0,1], index_col=[0,1]).round(3))
pd.read_csv(f'{R}/e11_erp_sensitivity_milp_vs_ranker.csv').round(4)